# 09 — Build the approved batch training dataset

Objective: combine reviewed human-derived robot recordings into one LeRobot dataset, with one episode per recording. Uses the existing notebook-06 format: two upright RGB cameras, 8D state, 7D executed actions, 20 Hz. Keeps every frame through release and withdrawal. No training or upload occurs here.

Run in the Colab environment where LeRobot is installed. Export uses CPU and local disk; a GPU is not required. All approved episodes are training data for the fixed-scene proof of concept; this is not a held-out generalization benchmark. Do not change approvals or source files during export.


In [ ]:
!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -e ".[libero]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, importlib.metadata
import numpy as np
import h5py
from scipy.spatial.transform import Rotation
from lerobot.datasets.lerobot_dataset import LeRobotDataset

project = Path('/content/drive/MyDrive/Projects/humanoid-challenge')
approved_file = project/'data/robot/approved_batch.json'
approved = json.loads(approved_file.read_text())['episodes']
assert approved, 'Run export-list in notebook 08 after approving successful replays.'
assert len({e['demo'] for e in approved}) == len(approved)
assert len({e['episode'] for e in approved}) == len(approved)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

def validate_report(report):
    assert report['status']=='passed_automatic_checks'
    assert report['training_eligible'] and not report['visual_review_required']
    review = report['visual_review']
    assert review.get('decision')=='approve' or review.get('approved') is True
    assert report['final_task_success'] and report['success_last_second']
    assert report['release_sequence_completed'] and report['bowl_plate_contact']
    assert not report['finger_bowl_contact']

# Freeze the approved inputs and verify annotation/trajectory provenance before copying.
reports = []
for entry in approved:
    report = json.loads((project/entry['report']).read_text())
    validate_report(report)
    source = project/'data/processed'/entry['demo']
    for filename, key in [('human_events.json','events'),('marker_smoothed.csv','trajectory')]:
        actual = sha256_file(source/filename)
        assert actual == entry['source'][key] == report['source_hashes'][filename], (entry['demo'], filename, 'changed')
    reports.append(report)
print('Approved episodes:', len(approved))
print('Demonstrations:', [e['demo'] for e in approved])
print('Expected frames:', sum(r['steps'] for r in reports))
print('Local free disk (GB):', round(shutil.disk_usage('/content').free/1e9,1))
REPO_ID = 'local/humanoid_human_batch'
work_dir = Path('/content/humanoid_dataset_exports')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
work_dir.mkdir(parents=True,exist_ok=False)
dataset_root = work_dir/'lerobot_dataset'
fps = 20


In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation

def state9_to_state8(state):
    """XYZ, quaternion XYZW, two finger qpos -> XYZ, rotvec, two finger qpos."""
    state = np.asarray(state, dtype=np.float64)
    if state.shape[-1] != 9 or not np.isfinite(state).all():
        raise ValueError("Expected finite state with last dimension 9")

    quaternions = state[..., 3:7]
    norms = np.linalg.norm(quaternions, axis=-1)
    if not np.allclose(norms, 1, atol=1e-3):
        raise ValueError("Quaternion norms are not close to one")

    normalized = quaternions / norms[..., None]
    rotvec = Rotation.from_quat(normalized.reshape(-1, 4)).as_rotvec()
    rotvec = rotvec.reshape(state.shape[:-1] + (3,))
    return np.concatenate([state[..., :3], rotvec, state[..., 7:9]], axis=-1).astype(np.float32)


def raw_libero_to_policy_features(ob):
    """Use for raw simulator observations at evaluation; images not yet tensorized."""
    raw_state = np.r_[ob["robot0_eef_pos"], ob["robot0_eef_quat"], ob["robot0_gripper_qpos"]]
    return {
        "observation.state": state9_to_state8(raw_state),
        "observation.images.image": np.ascontiguousarray(ob["agentview_image"][::-1]),
        "observation.images.image2": np.ascontiguousarray(ob["robot0_eye_in_hand_image"][::-1]),
    }


## Convert locally
Copy and convert one HDF5 at a time. Images are already upright: do not flip again. Source success signals do not terminate episodes; save_episode defines each boundary. This may take a while and needs enough local disk for the image dataset and ZIP archive.

In [ ]:
features = {
    "observation.images.image": {
        "dtype": "image", "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.images.image2": {
        "dtype": "image", "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.state": {
        "dtype": "float32", "shape": (8,),
        "names": ["eef_x", "eef_y", "eef_z", "rotvec_x", "rotvec_y", "rotvec_z", "finger_left", "finger_right"],
    },
    "action": {
        "dtype": "float32", "shape": (7,),
        "names": ["delta_x", "delta_y", "delta_z", "delta_rot_x", "delta_rot_y", "delta_rot_z", "gripper"],
    },
}

writer = LeRobotDataset.create(
    repo_id=REPO_ID,
    root=dataset_root,
    fps=fps,
    robot_type="libero_panda",
    features=features,
    use_videos=False,
)

references = []
provenance = []
offset = 0
conversion_finished = False
try:
    for episode_index, (entry, report) in enumerate(zip(approved, reports)):
        local_h5 = work_dir/'source_episode.h5'
        print(f"Copying {entry['demo']} ({episode_index+1}/{len(approved)})", flush=True)
        shutil.copy2(project/entry['episode'], local_h5)
        source_hash = sha256_file(local_h5)
        with h5py.File(local_h5,'r') as f:
            n = int(f.attrs['committed_steps'])
            assert n == report['steps'] and n > 0
            assert float(f.attrs['fps']) == fps
            assert bool(f.attrs['automatic_checks_passed'])
            language = f.attrs['language']
            if isinstance(language,bytes): language = language.decode()
            for key in ['agentview_image','wrist_image','state','action','phase','timestamp_s','success_after','done_after']:
                assert len(f[key]) == n, (entry['demo'],key,'incomplete recording')
            assert f['state'].shape == (n,9) and f['action'].shape == (n,7)
            for camera in ['agentview_image','wrist_image']:
                assert f[camera].shape == (n,256,256,3) and f[camera].dtype == np.uint8
            states = state9_to_state8(f['state'][:])
            actions = f['action'][:].astype(np.float32)
            assert np.isfinite(actions).all() and np.max(np.abs(actions)) <= 1.000001
            np.testing.assert_allclose(f['timestamp_s'][:],np.arange(n)/fps,atol=1e-6)
            assert np.all(f['success_after'][-fps:])
            phases = f['phase'].asstr()[:]
            assert 'release' in phases and 'withdraw' in phases
            samples = {}
            for i in range(n):
                image = f['agentview_image'][i]
                wrist = f['wrist_image'][i]
                writer.add_frame({'observation.images.image':image,
                                  'observation.images.image2':wrist,
                                  'observation.state':states[i].copy(),
                                  'action':actions[i].copy(), 'task':language})
                if i in {0,n//2,n-1}:
                    samples[i] = dict(state=states[i].copy(), action=actions[i].copy(),
                                      image=image.copy(), wrist=wrist.copy())
                if (i+1)%500 == 0: print(entry['demo'],i+1,'/',n,flush=True)
            writer.save_episode()
            references.append(dict(start=offset,n=n,language=language,samples=samples))
            provenance.append(dict(episode_index=episode_index, demo=entry['demo'],
                                   dataset_from_index=offset,dataset_to_index=offset+n,
                                   frames=n, source_h5=entry['episode'],source_h5_sha256=source_hash,
                                   source=entry['source'],source_report=report))
            offset += n
        local_h5.unlink()  # Only the temporary copy; source remains on Drive.
    conversion_finished = True
finally:
    writer.finalize()
assert conversion_finished
print('Converted',len(provenance),'episodes and',offset,'frames.')


## Verify episode boundaries and sample read-back
Checks first/middle/last samples from every episode, exact image reconstruction, and padding of action chunks at every episode end. A failure stops publication of the archive.

In [ ]:
dataset = LeRobotDataset(repo_id=REPO_ID, root=dataset_root)
assert len(dataset)==offset
assert dataset.meta.total_episodes==len(references)
assert dataset.meta.fps==fps
assert 'action' in dataset.meta.stats and 'observation.state' in dataset.meta.stats
chunk_size = 50
chunked = LeRobotDataset(repo_id=REPO_ID,root=dataset_root,
                         delta_timestamps={'action':[i/fps for i in range(chunk_size)]})
for episode_index, ref in enumerate(references):
    for local_index, expected in ref['samples'].items():
        item=dataset[ref['start']+local_index]
        assert int(item['episode_index'])==episode_index
        assert int(item['frame_index'])==local_index
        assert item['task']==ref['language']
        np.testing.assert_allclose(item['observation.state'].numpy(),expected['state'],atol=1e-6)
        np.testing.assert_allclose(item['action'].numpy(),expected['action'],atol=1e-7)
        for feature,key in [('observation.images.image','image'),('observation.images.image2','wrist')]:
            image=item[feature]
            assert tuple(image.shape)==(3,256,256)
            recovered=(image.permute(1,2,0).numpy()*255).round().astype(np.uint8)
            np.testing.assert_array_equal(recovered,expected[key])
    last=chunked[ref['start']+ref['n']-1]
    assert tuple(last['action'].shape)==(chunk_size,7)
    np.testing.assert_allclose(last['action'].numpy(),
        np.repeat(ref['samples'][ref['n']-1]['action'][None,:],chunk_size,axis=0),atol=1e-7)
    mask=last['action_is_pad'].numpy()
    assert mask.shape==(chunk_size,) and not mask[0] and mask[1:].all()
print('Read-back and boundary padding verified for every episode.')


## Archive and save to Drive
Keep the printed local root while this runtime remains connected. After a restart, restore the ZIP locally before training. This export uses all approved episodes for training; evaluate policy success separately in closed-loop simulation.

In [ ]:
preprocessing_source = 'import numpy as np\nfrom scipy.spatial.transform import Rotation\n\n\ndef state9_to_state8(state):\n    """XYZ, quaternion XYZW, two finger qpos -> XYZ, rotvec, two finger qpos."""\n    state = np.asarray(state, dtype=np.float64)\n    if state.shape[-1] != 9 or not np.isfinite(state).all():\n        raise ValueError("Expected finite state with last dimension 9")\n    quaternions = state[..., 3:7]\n    norms = np.linalg.norm(quaternions, axis=-1)\n    if not np.allclose(norms, 1, atol=1e-3):\n        raise ValueError("Quaternion norms are not close to one")\n    normalized = quaternions / norms[..., None]\n    rotvec = Rotation.from_quat(normalized.reshape(-1, 4)).as_rotvec()\n    rotvec = rotvec.reshape(state.shape[:-1] + (3,))\n    return np.concatenate([state[..., :3], rotvec, state[..., 7:9]], axis=-1).astype(np.float32)\n\n\ndef raw_libero_to_policy_features(ob):\n    """Use for raw simulator observations at evaluation; images not yet tensorized."""\n    raw_state = np.r_[ob["robot0_eef_pos"], ob["robot0_eef_quat"], ob["robot0_gripper_qpos"]]\n    return {\n        "observation.state": state9_to_state8(raw_state),\n        "observation.images.image": np.ascontiguousarray(ob["agentview_image"][::-1]),\n        "observation.images.image2": np.ascontiguousarray(ob["robot0_eye_in_hand_image"][::-1]),\n    }\n'
(dataset_root/'pilot_preprocessing.py').write_text(preprocessing_source)
manifest = dict(repo_id=REPO_ID,episodes=len(provenance),frames=offset,fps=fps,
                lerobot_version=importlib.metadata.version('lerobot'),sources=provenance,
                state_layout='xyz + scipy principal rotation vector (radians) + two finger qpos',
                images='upright RGB; lossless image features; source HDF5 unchanged',
                actions='normalized relative OSC_POSE commands, unchanged',
                purpose='fixed-scene proof of concept; all approved episodes used for training',
                readback_verified=True,episode_boundary_padding_verified=True)
(dataset_root/'conversion_manifest.json').write_text(json.dumps(manifest,indent=2))
archive=Path(shutil.make_archive(str(work_dir/'lerobot_dataset'),'zip',root_dir=dataset_root))
export_dir=project/'data/lerobot_exports'/work_dir.name
export_dir.mkdir(parents=True,exist_ok=False)
shutil.copy2(archive,export_dir/archive.name)
assert sha256_file(export_dir/archive.name)==sha256_file(archive)
shutil.copy2(dataset_root/'conversion_manifest.json',export_dir/'conversion_manifest.json')
print('Verified Drive archive:',export_dir/archive.name)
print('Local training root:',dataset_root)
print('Training repo_id:',REPO_ID)
print('Episodes:',len(provenance),'Frames:',offset)
